# Yggdrasil Research Worker R1 — Colab

Disposable compute for Yggdrasil only. CKB-plane remains the sole orchestration, acceptance, promotion, and runtime authority. This notebook does not choose successors or write to repositories.


In [ ]:
import datetime, hashlib, json, os, pathlib, platform, re, shutil, stat, subprocess, sys, zipfile
from google.colab import files

ROOT = pathlib.Path("/content/ckb-colab-r1")
ALLOWED_PROJECTS = {"Yggdrasil"}
ALLOWED_EXECUTABLES = {"python", "python3", "go"}
SENSITIVE_ENV = {
    "GITHUB_TOKEN","GH_TOKEN","OPENAI_API_KEY","COINBASE_API_KEY","COINBASE_API_SECRET",
    "CKB_OPERATOR_TOKEN","CKB_TOKEN","AWS_ACCESS_KEY_ID","AWS_SECRET_ACCESS_KEY","AZURE_CLIENT_SECRET"
}

def utc_now():
    return datetime.datetime.now(datetime.timezone.utc).isoformat()

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def safe_extract(zip_path, dest):
    dest=pathlib.Path(dest).resolve(); dest.mkdir(parents=True, exist_ok=True)
    prefix=str(dest)+os.sep
    with zipfile.ZipFile(zip_path) as z:
        for info in z.infolist():
            mode=(info.external_attr >> 16) & 0o170000
            if mode == stat.S_IFLNK:
                raise RuntimeError(f"symlink forbidden: {info.filename}")
            out=(dest / info.filename).resolve()
            if str(out) != str(dest) and not str(out).startswith(prefix):
                raise RuntimeError(f"zip path escape: {info.filename}")
            if info.is_dir():
                out.mkdir(parents=True, exist_ok=True); continue
            out.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as src, open(out,"xb") as dst:
                shutil.copyfileobj(src,dst)

def validate_job(job_root):
    mpath=job_root/"manifest.json"
    m=json.loads(mpath.read_text())
    assert m["schema"]=="ckb-plane.colab-research-job.v1"
    assert m["project"] in ALLOWED_PROJECTS
    assert re.fullmatch(r"[0-9a-f]{40}",m["source_commit"])
    cmd=m["command"]; assert isinstance(cmd,list) and 1 <= len(cmd) <= 64
    assert cmd[0].lower() in ALLOWED_EXECUTABLES
    for key,value in m["authority"].items():
        assert value is False, f"authority must be false: {key}"
    control=m["control"]
    for rel,key in [(control["preregistration_path"],"preregistration_sha256"),(control["seed_manifest_path"],"seed_manifest_sha256")]:
        p=(job_root/rel).resolve(); assert p.is_file(); assert sha256_file(p)==control[key]
    declared={}
    for f in m["payload_files"]:
        rel=f["path"]; assert rel.startswith("payload/")
        p=(job_root/rel).resolve(); assert p.is_file()
        assert sha256_file(p)==f["sha256"] and p.stat().st_size==f["bytes"]
        declared[rel]=f["sha256"]
    actual={str(p.relative_to(job_root)).replace(os.sep,"/") for p in (job_root/"payload").rglob("*") if p.is_file()}
    assert actual==set(declared), "payload file set mismatch"
    return m, sha256_file(mpath)

def run_capture(argv,cwd,timeout,env):
    return subprocess.run(argv,cwd=str(cwd),env=env,stdout=subprocess.PIPE,stderr=subprocess.PIPE,timeout=timeout,shell=False)

print("YGGDRASIL_COLAB_WORKER_R1_READY")


In [ ]:
uploaded=files.upload()
if len(uploaded)!=1:
    raise RuntimeError("upload exactly one CKB Colab job ZIP")
name,data=next(iter(uploaded.items()))
if ROOT.exists():
    shutil.rmtree(ROOT)
job_root=ROOT/"job"; work=ROOT/"work"; evidence_root=ROOT/"evidence"
ROOT.mkdir(parents=True)
job_zip=ROOT/"job.zip"; job_zip.write_bytes(data)
safe_extract(job_zip,job_root)
manifest,job_manifest_sha256=validate_job(job_root)
for key in SENSITIVE_ENV:
    if os.environ.get(key):
        raise RuntimeError(f"authority credential present in environment: {key}")
print(json.dumps({
    "validated":True,
    "project":manifest["project"],
    "experiment_id":manifest["experiment_id"],
    "source_commit":manifest["source_commit"],
    "job_manifest_sha256":job_manifest_sha256,
    "accelerator_requested":manifest["accelerator"]
},indent=2))


In [ ]:
shutil.copytree(job_root/"payload",work)
# Dependency hydration is bounded to standard lock mechanisms; no arbitrary setup shell is accepted.
req=work/"requirements.lock"
if req.exists():
    lines=[x.strip() for x in req.read_text().splitlines() if x.strip() and not x.lstrip().startswith("#")]
    if not lines or any("--hash=" not in x for x in lines):
        raise RuntimeError("requirements.lock must use pip hashes for every requirement")
    subprocess.run([sys.executable,"-m","pip","install","--require-hashes","-r",str(req)],check=True,shell=False)
if (work/"go.mod").exists():
    if not (work/"go.sum").exists():
        raise RuntimeError("go.mod present without go.sum")
    subprocess.run(["go","mod","download"],cwd=str(work),check=True,shell=False)
print("DEPENDENCIES_READY")


In [ ]:
env=os.environ.copy()
for key in SENSITIVE_ENV:
    env.pop(key,None)
env["CKB_COLAB_PROJECT"]=manifest["project"]
env["CKB_COLAB_EXPERIMENT_ID"]=manifest["experiment_id"]
env["CKB_COLAB_SOURCE_COMMIT"]=manifest["source_commit"]
env["PYTHONDONTWRITEBYTECODE"]="1"
started=utc_now(); status="failed"; exit_code=None
stdout=b""; stderr=b""
try:
    p=run_capture(manifest["command"],work,int(manifest["budget"]["max_seconds"]),env)
    stdout,stderr=p.stdout,p.stderr; exit_code=p.returncode
    status="completed" if p.returncode==0 else "failed"
except subprocess.TimeoutExpired as e:
    status="timed_out"; stdout=e.stdout or b""; stderr=e.stderr or b""
finished=utc_now()

evidence_root.mkdir(parents=True,exist_ok=True)
(evidence_root/"stdout.log").write_bytes(stdout)
(evidence_root/"stderr.log").write_bytes(stderr)
collected=[]
collect_candidates=[]
for rel in ["classification.json","result.json","results.json","metrics.json","metrics.csv","receipt.json"]:
    p=work/rel
    if p.is_file(): collect_candidates.append(p)
results_dir=work/"results"
if results_dir.is_dir():
    collect_candidates.extend([p for p in results_dir.rglob("*") if p.is_file()])
for p in sorted(set(collect_candidates)):
    rel=p.relative_to(work)
    out=evidence_root/"collected"/rel
    out.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(p,out)
    evidence_rel=str(out.relative_to(evidence_root)).replace(os.sep,"/")
    collected.append({"path":evidence_rel,"sha256":sha256_file(out),"bytes":out.stat().st_size})

def version(cmd):
    try:
        p=subprocess.run(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=15,shell=False)
        return p.stdout.decode("utf-8","replace").strip()
    except Exception as e:
        return f"unavailable:{type(e).__name__}"

evidence={
    "schema":"ckb-plane.colab-research-evidence.v1",
    "project":manifest["project"],
    "experiment_id":manifest["experiment_id"],
    "source_commit":manifest["source_commit"],
    "job_manifest_sha256":job_manifest_sha256,
    "status":status,
    "exit_code":exit_code,
    "started_at_utc":started,
    "finished_at_utc":finished,
    "command":manifest["command"],
    "stdout_sha256":sha256_file(evidence_root/"stdout.log"),
    "stderr_sha256":sha256_file(evidence_root/"stderr.log"),
    "collected_files":collected,
    "environment":{
        "python":sys.version,
        "platform":platform.platform(),
        "go":version(["go","version"]),
        "gpu":version(["nvidia-smi","--query-gpu=name,driver_version","--format=csv,noheader"])
    },
    "authority":{
        "ckb_queue_mutated":False,
        "accepted_ref_mutated":False,
        "production_authority_changed":False,
        "broker_accessed":False,
        "repository_write_credentials_present":False,
        "network_isolation_proven":False
    }
}
(evidence_root/"evidence.json").write_text(json.dumps(evidence,sort_keys=True,separators=(",",":")))
out_zip=ROOT/f'{manifest["experiment_id"]}-evidence.zip'
with zipfile.ZipFile(out_zip,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(evidence_root.rglob("*")):
        if p.is_file(): z.write(p,p.relative_to(evidence_root))
print(json.dumps(evidence,indent=2))
files.download(str(out_zip))
